# 🎬 Обучение Qwen 2.5 Режиссёра v2 (QLoRA, Unsloth, GGUF)

Этот ноутбук обучает локальную модель **Qwen2.5-7B-Instruct** роли монтажного режиссёра (генерация схемы раскадровки `beats`) в соответствии с Планом проекта (Фаза Ф7):
- Обучение только на ответах ассистента (`train_on_responses_only`)
- Контекст 8192 токена (`num_ctx: 8192`)
- Разделение train/val выборки для контроля `eval_loss`
- Экспорт адаптера в **GGUF (q4_k_m)** и генерация `Modelfile` для моментального создания модели в **Ollama**

In [ ]:
# 1. Установка Unsloth и зависимостей (~1 мин)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 2. Загрузка данных (загрузите director_train.jsonl и director_val.jsonl в Colab)
import os
from datasets import load_dataset

train_path = "director_train.jsonl"
val_path = "director_val.jsonl"

if not os.path.exists(train_path):
    print(f"⚠️ Загрузите {train_path} и {val_path} через панель файлов слева (Files -> Upload)")
else:
    train_ds = load_dataset("json", data_files=train_path, split="train")
    val_ds = load_dataset("json", data_files=val_path, split="train") if os.path.exists(val_path) else None
    print(f"Загружено: train={len(train_ds)}, val={len(val_ds) if val_ds else 0}")

In [ ]:
# 3. Инициализация базовой модели Qwen2.5-7B (4-bit)
import torch
from unsloth import FastLanguageModel

max_seq_length = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Настраиваем LoRA адаптер (r=16, alpha=32)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 4. Форматирование промптов и обучение ТОЛЬКО на ответах (train_on_responses_only)
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convs]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True)
if val_ds:
    val_ds = val_ds.map(formatting_prompts_func, batched=True)

In [ ]:
# 5. Запуск обучения (SFTTrainer)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="epoch" if val_ds else "no",
        save_strategy="epoch" if val_ds else "no",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="out_director_checkpoints",
        report_to="none",
    ),
)

# Маскируем токены пользователя и системы, обучаем только ответы режиссёра
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

In [ ]:
# 6. Экспорт в GGUF для Ollama (q4_k_m)
out_dir = "out_director"
os.makedirs(out_dir, exist_ok=True)

print("Экспортируем модель в формат GGUF q4_k_m...")
model.save_pretrained_gguf(out_dir, tokenizer, quantization_method="q4_k_m")

# Находим экспортированный gguf
gguf_files = [f for f in os.listdir(out_dir) if f.endswith(".gguf")]
gguf_name = gguf_files[0] if gguf_files else "qwen2.5-director-v1.Q4_K_M.gguf"

modelfile_content = f"""FROM ./{gguf_name}
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 8192
PARAMETER stop \"<|im_start|>\"
PARAMETER stop \"<|im_end|>\"
"""

with open(os.path.join(out_dir, "Modelfile"), "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print(f"\nГотово! Файлы сохранены в папке {out_dir}/")
print("1. Скачайте папку out_director на локальный компьютер в training/out_director")
print("2. Выполните в терминале:")
print("   cd training/out_director")
print("   ollama create qwen2.5-director-v1 -f Modelfile")
print("3. В .env установите: OLLAMA_DIRECTOR_MODEL=qwen2.5-director-v1")